# 06.11 - RNNs, LSTMs, GRUs

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Recurrent Neural Networks process sequential data by maintaining a hidden state that carries information across time steps. LSTMs and GRUs fix the vanishing-gradient problem of vanilla RNNs.

## 2. Why Does This Matter?

Many problems involve sequences: text, time series, speech, music, genomics. MLPs/CNNs treat each input independently; sequences have temporal dependencies RNNs capture.

## 3. Prerequisites

- Units 06.4, 06.8 (PyTorch, training loops), basic linear algebra

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build and train RNNs, LSTMs, GRUs in PyTorch
- Explain gating mechanisms and vanishing gradients
- Choose the right recurrent architecture

## 5. Mental Model

An RNN reads a sequence word-by-word, updating a hidden state. An LSTM adds a 'notebook' (cell state) to remember important info and forget irrelevant details across long passages.

`h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)`

> Synthetic sequence task only (no real dataset download).


## 6. Backend


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42); np.random.seed(42)
print("PyTorch version:", torch.__version__)


## 7. Build a Synthetic Sequence Task

Binary sequence classification: the class depends on the FIRST element of the sequence (requires remembering across a gap).


In [ ]:
def make_sequences(n=600, seq_len=12, noise=0.1):
    X = np.random.randn(n, seq_len, 1) * noise
    y = np.zeros(n, dtype=np.int64)
    first = np.random.randint(0, 2, n)          # signal = first element sign
    for i in range(n):
        X[i, 0, 0] += 1.0 if first[i] == 1 else -1.0
        y[i] = int(X[i, 0, 0] > 0)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X, y = make_sequences()
split = int(0.7 * len(X))
Xtr, ytr = X[:split], y[:split]
Xte, yte = X[split:], y[split:]
print("Train:", Xtr.shape, "Test:", Xte.shape, "(seq_len=12, input_dim=1)")


## 8. RNN / LSTM / GRU Basics

Instantiate each and inspect output shape: `(batch, seq, hidden)` with `batch_first=True`.


In [ ]:
batch, seq_len, dim = 4, 10, 1
x = torch.randn(batch, seq_len, dim)
print("Input:", tuple(x.shape))
for name, layer in [('RNN', nn.RNN(dim, 16, batch_first=True)),
                    ('LSTM', nn.LSTM(dim, 16, batch_first=True)),
                    ('GRU', nn.GRU(dim, 16, batch_first=True))]:
    out, h = layer(x)
    print(f"{name:5s}: output={tuple(out.shape)}")
print("\nout[:, -1, :] holds the final hidden state per sequence — used for classification.")


## 9. Build a Full Model Wrapper and Train

Train each architecture on the same classification task and compare.


In [ ]:
class SeqClf(nn.Module):
    def __init__(self, kind):
        super().__init__()
        if kind == 'rnn':
            self.core = nn.RNN(1, 16, batch_first=True)
        elif kind == 'lstm':
            self.core = nn.LSTM(1, 16, batch_first=True)
        else:
            self.core = nn.GRU(1, 16, batch_first=True)
        self.head = nn.Linear(16, 2)
        self.kind = kind
    def forward(self, x):
        out, h = self.core(x)
        return self.head(out[:, -1, :])  # last time step

def train_clf(kind, epochs=25):
    m = SeqClf(kind)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(m.parameters(), lr=0.005)
    for _ in range(epochs):
        opt.zero_grad(); crit(m(Xtr), ytr).backward(); opt.step()
    with torch.no_grad():
        acc = (m(Xte).argmax(dim=1) == yte).float().mean().item()
    return acc

print("Architecture comparison (same data, 25 epochs):")
accs = {}
for k in ['rnn', 'lstm', 'gru']:
    accs[k] = train_clf(k)
    print(f"  {k:5s} test acc = {accs[k]:.3f}")
print("\nAll can solve this short task; differences widen on long sequences.")


## 10. Vanishing Gradients: RNN vs LSTM over Long Sequences

Increase sequence length so the needed signal is far from the output — LSTMs retain it better.


In [ ]:
def make_long(seq_len=80):
    X = np.random.randn(400, seq_len, 1) * 0.1
    y = np.zeros(400, dtype=np.int64)
    first = np.random.randint(0, 2, 400)
    X[:, 0, 0] += np.where(first == 1, 1.0, -1.0)
    y = (X[:, 0, 0] > 0).astype(np.int64)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

Xl, yl = make_long(80)
sl = int(0.7 * len(Xl))
def accuracy_long(kind):
    m = SeqClf(kind)  # input dim 1 still
    crit = nn.CrossEntropyLoss(); opt = optim.Adam(m.parameters(), lr=0.003)
    for _ in range(30):
        opt.zero_grad(); crit(m(Xl[:sl]), yl[:sl]).backward(); opt.step()
    with torch.no_grad():
        return (m(Xl[sl:]).argmax(dim=1) == yl[sl:]).float().mean().item()

print("Long-sequence task (len=80), signal at position 0:")
for k in ['rnn', 'lstm', 'gru']:
    print(f"  {k:5s} test acc = {accuracy_long(k):.3f}")
print("\nLSTM/GRU usually beat vanilla RNN here — the gating retains the early signal.")


## 11. Gating Comparison Table

| Architecture | Parameters | Memory | Speed | Use When |
|---|---|---|---|---|
| Vanilla RNN | Fewest | Lowest | Fastest | Short seq, simple tasks |
| GRU | Few | Low | Fast | Medium seq, general purpose |
| LSTM | Most | High | Moderate | Long seq, complex deps |
| Bidirectional | 2x | 2x | Slower | Full context available |

## 12. Common Mistakes / Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Loss NaN | Exploding gradients | Check grad norms | Clip gradients, lower LR |
| Can't capture long deps | vanilla RNN | Switch arch | LSTM/GRU |
| Memory on long seq | Too long | Check length | Packing, shorter windows |
| Very slow | No batching | Check loader | Batch, shorter windows |

- Detach hidden states between independent batches when doing truncated BPTT.
- Real-time requires unidirectional; bidirectional needs full context.

## 13. Real-World Considerations

- A stock predictor with 2-layer LSTM (128 hidden) processes 60-day windows and outputs up/down.

## 14. When NOT to Use

- Very long sequences where attention/transformers are superior (next unit).

## 15. Challenge

Add a bidirectional LSTM and compare to unidirectional on the long task.


In [ ]:
# Challenge: bidirectional LSTM
class BiSeq(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = nn.LSTM(1, 16, bidirectional=True, batch_first=True)
        self.head = nn.Linear(16*2, 2)
    def forward(self, x):
        out, _ = self.core(x)
        return self.head(out[:, -1, :])

m = BiSeq(); crit = nn.CrossEntropyLoss(); opt = optim.Adam(m.parameters(), lr=0.003)
for _ in range(30):
    opt.zero_grad(); crit(m(Xl[:sl]), yl[:sl]).backward(); opt.step()
with torch.no_grad():
    acc = (m(Xl[sl:]).argmax(dim=1) == yl[sl:]).float().mean().item()
print(f"Bidirectional LSTM test acc: {acc:.3f}")
print("Bidirectional sees both directions — good when full sequence is available.")


## 16. Closed-Book Recall

Without looking back:

1. How does the LSTM cell state solve vanishing gradients?
2. Difference between GRU and LSTM gates?
3. When bidirectional vs unidirectional?
4. Why detach hidden states?

## 17. Teach-Back Questions

Explain to another person:

- How an RNN carries memory through a sequence.
- Why LSTMs beat vanilla RNNs on long sequences.

## 18. Summary

You built and trained RNN, LSTM, and GRU classifiers on synthetic sequence data and compared length handling.

## 19. Further Experiment

- Add gradient clipping to the long RNN.
- Truncated backprop with hidden-state detach.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
